# 03 — 实时信号看板

每次运行本 notebook 即可查看：
- **当前费率快照**：11 只合约的实时费率 + 下期预测
- **当前信号**：满足阈值的合约及建议方向
- **信号变动**：与上次运行相比，新开/平仓/换向事件
- **历史走势**：最近 30 天费率走势图

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import datetime, timezone

from src.monitor.scanner import scan, load_last_snapshot, diff_snapshots
from src.data.binance import load_all, EQUITY_SYMBOLS

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print(f'看板时间: {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")}')

## 1. 实时扫描

In [ ]:
# 加载上次快照
prev_snapshot = load_last_snapshot()

# 拉取实时数据并生成新快照（threshold=0.0001 即 0.01%/8h = 年化11%）
curr_snapshot = scan(threshold=0.0001, verbose=True)

## 2. 信号变动事件

In [ ]:
events = diff_snapshots(prev_snapshot, curr_snapshot)
if events:
    print(f'【变动事件】（{len(events)} 项）')
    for e in events:
        print(f'  {e}')
else:
    print('无变动 — 信号与上次相同')

## 3. 当前费率条形图

In [ ]:
rates = curr_snapshot.get('rates', {})
if rates and 'current_ann' in rates:
    rate_df = pd.DataFrame(rates)
    rate_df.index = rate_df.index.str.replace('USDT', '')
    rate_df = rate_df.sort_values('current_ann', ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['green' if v > 0 else 'red' for v in rate_df['current_ann']]
    ax.barh(rate_df.index, rate_df['current_ann'], color=colors, alpha=0.8, label='当期')
    ax.barh(rate_df.index, rate_df['next_ann'],    color=colors, alpha=0.35, 
            edgecolor=colors, linewidth=1, linestyle='--', label='下期预测')
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('年化费率 %/年')
    ax.set_title(f'Binance TradFi 实时费率  ({curr_snapshot["ts"][:16]} UTC)')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('rates 数据不可用')

## 4. 历史费率走势（最近 30 天）

In [ ]:
# 加载历史数据（从缓存，若缓存不存在则拉取）
wide = load_all()
# 最近30天
recent = wide.last('30D')
ann_recent = recent * 3 * 365 * 100

# 有效信号（有数据的合约）
active_syms = ann_recent.columns[(ann_recent != 0).sum() > 5]

n = len(active_syms)
ncols = 3
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows), sharex=True)
axes_flat = axes.flatten() if n > 1 else [axes]

for idx, sym in enumerate(active_syms):
    ax = axes_flat[idx]
    series = ann_recent[sym].dropna()
    colors = ['green' if v > 0 else 'red' for v in series.values]
    ax.bar(series.index, series.values, width=pd.Timedelta('7h'), color=colors, alpha=0.7)
    ax.axhline(0, color='black', lw=0.5)
    # 入场阈值线
    thr = 0.0001 * 3 * 365 * 100
    ax.axhline( thr, color='blue', ls=':', lw=0.8, alpha=0.6)
    ax.axhline(-thr, color='blue', ls=':', lw=0.8, alpha=0.6)
    ax.set_title(sym.replace('USDT', ''), fontsize=9)
    ax.tick_params(axis='x', rotation=30, labelsize=7)

for ax in axes_flat[n:]:
    ax.set_visible(False)

plt.suptitle('最近30天费率走势（蓝点线=入场阈值）', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

## 5. IBKR 执行建议

In [ ]:
from src.execution.ibkr import TICKER_MAP

signals = curr_snapshot.get('signals', {})
if not signals:
    print('当前无信号，无需操作')
else:
    print('【建议操作】（假设名义本金 $10,000，需连接 IBKR TWS 执行）')
    print(f'{"合约":<10} {"Binance方向":<15} {"IBKR合成":<25} {"年化费率"}')
    print('-' * 65)
    for sym, (direction, weight, rate_ann) in sorted(
        signals.items(), key=lambda x: -abs(x[1][2])
    ):
        ticker = TICKER_MAP.get(sym, '?')
        if direction > 0:
            binance_side = '做空永续 ↓'
            ibkr_side    = f'买Call+卖Put ({ticker})'
        else:
            binance_side = '做多永续 ↑'
            ibkr_side    = f'卖Call+买Put ({ticker})'
        label = sym.replace('USDT', '')
        print(f'  {label:<8} {binance_side:<15} {ibkr_side:<25} {rate_ann:+.1f}%/年')

print('\n⚠️ 实盘前请用 IBKRExecutor(..., dry_run=True) 验证下单逻辑')

## 6. 运行监控脚本（可选）

在终端中运行以下命令启动每8小时自动扫描：

```bash
# 立即扫描一次
uv run python scripts/monitor_loop.py --once

# 持续运行（每8小时自动扫描，Ctrl+C 停止）
uv run python scripts/monitor_loop.py

# 降低阈值（捕捉更多信号）
uv run python scripts/monitor_loop.py --once --threshold 0.00005
```